# PE6201 A2 - Problem A Agent

This notebook is a thin local entry point for the current implementation.

It assumes the repository contains:

```text
src/
- prompt.py
- agent_core.py
- harness.py
- evaluation.py
- tools.py
- guardrails.py
- backends.py
- config.py
```

Top-level runner:

```text
run_eval.py
```

Current design:
- 5 agent-visible tools: `get_claim`, `lookup_policy`, `check_coverage`, `get_preauthorisation`, `issue_decision_letter`
- 4 internal helpers: `check_duplicate`, `check_hospital_panel`, `check_missing_documents`, `already_decided`
- single-agent ReAct loop in `src.agent_core`
- prompt text in `src.prompt`
- evaluation/run aggregation in `src.harness`
- one model turn = exactly one JSON object
- model must never invent or simulate tool observations


## 1. Environment setup

If you are using Colab, clone the repo first.  
If you are running locally from the repo root, skip the clone cell.

In [ ]:
# OPTIONAL — Colab only
!git clone https://github.com/Tonya0719/PE6201-A2-Group3.git
%cd /content/PE6201-A2-Group3

In [ ]:
# Install dependencies if needed.
# Uncomment in a fresh Colab/runtime.

%pip install -q openai python-dotenv

## 2. Imports

In [ ]:
import os
import json
import importlib
from pprint import pprint

from src import config
from src.agent_core import run_agent
from src.harness import print_d2c_report, run_d2c_comparison, run_evaluation, summarize_results
from src.prompt import build_system_prompt
from src.tools import get_tool_specs


## 3. Current configuration

For formal comparisons, keep the same prompt, eval set and code commit.  
Change only the intended experimental variable, such as `MODEL`, `TOOL_SPEC_VERSION`, or `PARALLEL_ENABLED`.

In [ ]:
print("PE6201 A2 - Problem A Agent")
print(f"BACKEND                 : {config.BACKEND}")
print(f"MODEL                   : {config.MODEL}")
print(f"TOOL_SPEC_VERSION       : {config.TOOL_SPEC_VERSION}")
print(f"PARALLEL_ENABLED        : {config.PARALLEL_ENABLED}")
print(f"MAX_TOOL_CALLS_PER_TURN : {config.MAX_TOOL_CALLS_PER_TURN}")
print(f"STEP_CAP                : {config.STEP_CAP}")
print(f"BUDGET_USD              : {config.BUDGET_USD}")
print(f"AUTONOMY                : {config.AUTONOMY}")
print("=" * 60)


## 4. Run one claim

Recommended development claims:
- `CLM-8842` — ordinary ACT, multi-line + preauth + excluded line
- `CLM-8888` — ASK, missing preauth
- `CLM-8910` — policy-level ESCALATE
- `CLM-8933` — duplicate early exit

With the latest design, expected path lengths are approximately:
- duplicate / hostile → **2 turns**
- policy-level escalation → **3 turns**
- ordinary ACT / ASK → **4 turns**

In [ ]:
CASE_ID = "CLM-8842"

result = run_agent(
    CASE_ID,
    approved_for_write=False,   # keep False when AUTONOMY="confirm"
)

print("\nRUN RESULT")
print(json.dumps(result, indent=2, ensure_ascii=False))

## 5. Optional: allow the gated write

Only use this when you explicitly want to test the local structured write.

With `AUTONOMY="confirm"`:
- `approved_for_write=False` → proposal is returned, write is blocked
- `approved_for_write=True` → runtime may execute `issue_decision_letter`

In [ ]:
# Uncomment to test the gated write.
#
# approved_result = run_agent(
#     CASE_ID,
#     approved_for_write=True,
# )
#
# print(json.dumps(approved_result, indent=2, ensure_ascii=False))

## 6. Inspect the tool trajectory

This cell makes the ReAct path easier to inspect during debugging and demos.

In [ ]:
def print_tool_trace(run_result):
    for step in run_result.get("tool_history", []):
        print(f"\nTURN {step.get('turn')}")
        print("Tool calls:")
        for call in step.get("tool_calls", []):
            print("  -", call.get("name"), call.get("arguments", {}))

        print("Observations:")
        for obs in step.get("observations", []):
            if obs.get("type") == "helper_context":
                print("  - helper_context:", obs)
            else:
                print("  -", obs.get("tool"), "ok=" + str(obs.get("ok")))
                pprint(obs.get("result"))

print_tool_trace(result)

The current `build_system_prompt(...)` should enforce:

1. ONE MODEL TURN = EXACTLY ONE JSON OBJECT
2. If the model returns `type="action"`, it must STOP
3. It must never generate Observation objects
4. It must never invent policy, preauthorisation, coverage, hospital, document, or duplicate facts
5. Facts not seen in real observations are unknown
6. Multiple independent tool calls may be returned inside one action
7. `get_claim` must come first; `lookup_policy` only after intake clears
8. `check_coverage` must complete before any `get_preauthorisation` request
9. `issue_decision_letter` is model-selected, and the runtime applies the gate immediately before that write
10. Final is bookkeeping only after the write observation exists

In [ ]:
tool_specs = get_tool_specs(config.TOOL_SPEC_VERSION)
prompt = build_system_prompt(tool_specs)

print(prompt[:5000])

## 8. Scripted evaluation

Use the scripted backend for deterministic, no-network regression checks.

This does **not** replace the required live-model evidence.

In [ ]:
# Save current backend, switch temporarily, then restore.
original_backend = config.BACKEND
config.BACKEND = "scripted"

try:
    scripted_results = run_evaluation()
    scripted_summary = summarize_results(scripted_results)

    print("SCRIPTED SUMMARY")
    print(json.dumps(scripted_summary, indent=2, ensure_ascii=False))
finally:
    config.BACKEND = original_backend

## 9. Live evaluation

For D5(b), use the same:
- evaluation set
- v2 prompt
- code commit
- tool set

Change only `MODEL`.

Do not run the full live battery repeatedly during development.

In [ ]:
# Example only — uncomment when ready for formal live evaluation.
#
# config.BACKEND = "live"
#
# live_results = run_evaluation()
# live_summary = summarize_results(live_results)
#
# print("LIVE SUMMARY")
# print(json.dumps(live_summary, indent=2, ensure_ascii=False))

## 10. D2(c) sequential vs batched comparison

For a controlled comparison:
- keep backend, model, prompt, eval cases and tool set fixed
- compare `MAX_TOOL_CALLS_PER_TURN=1` against `MAX_TOOL_CALLS_PER_TURN=None`
- report correctness, turns, model calls, tokens and cost
- `PARALLEL_ENABLED` is only physical concurrent execution inside an already-batched turn; it is not the D2(c) turn-collapse variable


In [ ]:
# D2(c) report helper. Uses scripted backend by default and restores config afterwards.
comparison = run_d2c_comparison(
    case_ids=["CLM-8842", "CLM-8960"],  # use None for the full evaluation set
    backend="scripted",                  # change to "live" for measured token/cost evidence
    approved_for_write=True,
    tool_spec_version=config.TOOL_SPEC_VERSION,
    negative_trials=1,
    ordinary_trials=1,
)

print_d2c_report(comparison)


## 11. D2(b) tool-spec v1 vs v2 comparison

Keep:
- same model
- same eval set
- same business logic
- same code commit

Change only `TOOL_SPEC_VERSION`.

In [ ]:
# Example experiment scaffold.
#
# original_spec = config.TOOL_SPEC_VERSION
#
# try:
#     config.TOOL_SPEC_VERSION = "v1"
#     v1_result = run_agent("CLM-8842")
#
#     config.TOOL_SPEC_VERSION = "v2"
#     v2_result = run_agent("CLM-8842")
#
#     print({
#         "v1": {
#             "turns": v1_result.get("turns"),
#             "input_tokens": v1_result.get("input_tokens"),
#             "output_tokens": v1_result.get("output_tokens"),
#             "cost": v1_result.get("cost"),
#             "decision": v1_result.get("decision"),
#         },
#         "v2": {
#             "turns": v2_result.get("turns"),
#             "input_tokens": v2_result.get("input_tokens"),
#             "output_tokens": v2_result.get("output_tokens"),
#             "cost": v2_result.get("cost"),
#             "decision": v2_result.get("decision"),
#         },
#     })
# finally:
#     config.TOOL_SPEC_VERSION = original_spec

## 12. Final run checklist

Before collecting formal evidence:
- confirm repository commit hash
- confirm `TOOL_SPEC_VERSION="v2"`
- confirm the correct eval set
- confirm clean state before every case
- record model + prompt version + date + trials
- preserve tool history, token usage, cost and guardrail events
- use the same code for scripted and live runs